<a href="https://colab.research.google.com/github/anurag-kandi01/ML-Repo/blob/main/AIML_Module_4_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4: Linear Classifiers & Gradient Descent

**Case Study: Predictive Modeling for Public Water Safety**

**Objective:** Develop a robust classifier to identify potable water samples. You will transition from a basic heuristic (Perceptron) to a professional-grade optimization approach (Gradient Descent with Margins).

# 1. Data Acquisition & Cleaning

In real-world data science, datasets are rarely perfect. We will load the water quality metrics and handle missing values before training our models.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset from a public raw GitHub URL
url = "https://raw.githubusercontent.com/nferran/tp_aprendizaje_de_maquina_I/main/water_potability.csv"
df = pd.read_csv(url)

# Step 1: Handling Missing Values
# Water sensors often fail, leaving NaNs. We will fill them with the mean of the column.
df.fillna(df.mean(), inplace=True)

# Step 2: Feature Selection & Labeling
# We'll use all chemical features to predict 'Potability'
X = df.drop('Potability', axis=1).values
y = df['Potability'].values

# Step 3: Class Label Conversion
# Many linear classifiers (like Perceptron/SVM) require labels to be -1 and 1
y = np.where(y == 0, -1, 1)

# Step 4: Train-Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Dataset Loaded: {X_train.shape[0]} training samples, {X_train.shape[1]} features.")

Dataset Loaded: 2620 training samples, 9 features.


# 2. Phase 1: The Heuristic Approach (Perceptron)

The **Perceptron** represents the earliest form of supervised learning. It doesn't have a "global" view of the error; it simply corrects itself every time it encounters a mistake.

**Task:** Implement the Perceptron Update Rule inside the training loop.

In [2]:
class WaterPerceptron:
    def __init__(self, lr=0.01, epochs=50):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0
        self.mistakes = []

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        for epoch in range(self.epochs):
            count = 0
            for i in range(len(y)):
                # TODO: Calculate the linear output (w * x + b)
                # prediction = ...

                # TODO: If prediction is a mistake (y * prediction <= 0):
                # Update weights: w = w + lr * y * x
                # Update bias: b = b + lr * y
                pass # remove this
            self.mistakes.append(count)

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)

# model_p = WaterPerceptron()
# model_p.fit(X_train, y_train)

# 3. Phase 2: Gradient Descent - Global Optimization

The Perceptron is unstable if the data isn't perfectly separable. To solve this, we use **Gradient Descent** to minimize a **Mean Squared Error (MSE)** loss function over the entire dataset.

**Task:** Implement the batch gradient calculation for weights and bias.

In [3]:
class GDWaterClassifier:
    def __init__(self, lr=0.001, epochs=500):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0
        self.cost_history = []

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        n = X.shape[0]

        for _ in range(self.epochs):
            # TODO: 1. Compute linear output: z = Xw + b
            # TODO: 2. Calculate gradients:
            # dw = (1/n) * X.T.dot(z - y)
            # db = (1/n) * sum(z - y)

            # TODO: 3. Update w and b: w = w - lr * dw
            pass

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)

# 4. Phase 3: Margin Classifiers & Hinge Loss

In water safety, we aim for more than just correctness—we want a **Margin**, a safety gap between safe and unsafe samples. This is achieved using **Hinge Loss** combined with **L2 Regularization**.

The loss function is defined as:

$$
\text{Loss} = \lambda \|w\|^2_2 + \sum_{i} \max(0, 1 - y_i (w^T x_i + b))
$$

### Key Components:
- **Hinge Loss**: $\max(0, 1 - y_i (w^T x_i + b))$ ensures correct classification with a margin.
- **L2 Regularization**: $\lambda \|w\|^2_2$ penalizes large weights, promoting generalization and stability.


In [4]:
class MarginWaterClassifier:
    def __init__(self, lr=0.001, lambda_param=0.01, epochs=500):
        self.lr = lr
        self.lambda_param = lambda_param
        self.epochs = epochs
        self.w = None
        self.b = 0

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        for _ in range(self.epochs):
            for i, x_i in enumerate(X):
                # TODO: Implement the Margin Condition check: y_i * (w * x_i + b) >= 1
                if False: # Replace False with condition
                    # Only Regularization update
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Update for weight (including Hinge Loss) and bias
                    # self.w -= self.lr * (2 * self.lambda_param * self.w - x_i * y[i])
                    # self.b -= self.lr * (-y[i])
                    pass

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)

# 5. Critical Analysis & Comparison

**Analysis Tasks:**
1. Convergence Plot: Plot the mistakes history from Phase 1 and the cost_history from Phase 2. Discuss why the Gradient Descent plot is smoother.
2. Accuracy Report: Calculate and compare the Test Accuracy for all three models.
3. Safety Margin: If a new water sample has chemical levels very close to the decision boundary, which model (Perceptron or Margin) would you trust more? Why?

1. Convergence Plot

The convergence plot is used to understand how the models learn as the number of iterations or epochs increases. In this experiment, we compare the mistakes history from Phase 1 with the cost history from Phase 2.

Perceptron – Mistakes History

In Phase 1, the Perceptron keeps track of the number of incorrectly classified samples during each epoch. The mistakes history represents how many training examples were misclassified at each iteration.

As training progresses, the number of mistakes generally decreases because the Perceptron updates its weights whenever it encounters a misclassified sample. If the data is linearly separable, the Perceptron can eventually reach a point where there are very few or even zero classification errors.

However, the Perceptron plot may not be perfectly smooth. The number of mistakes can increase, decrease, or remain unchanged between successive epochs. This happens because the Perceptron uses discrete classification decisions and updates its weights based on individual misclassified samples.

Gradient Descent – Cost History

In Phase 2, Gradient Descent records the value of the cost function after each iteration. The cost represents the overall error of the model.

As Gradient Descent updates the weights in the direction that reduces the cost, the cost generally decreases gradually toward a minimum.

The Gradient Descent curve is generally smoother than the Perceptron mistakes curve because Gradient Descent uses the gradient of a continuous loss function to make controlled updates to the weights. Instead of simply counting whether a prediction is correct or incorrect, it considers how large the prediction error is and adjusts the weights accordingly.

Why is the Gradient Descent plot smoother?

The main reasons are:

Continuous loss function: Gradient Descent works with a continuous cost/loss value.
Gradual weight updates: Weights are updated in small steps according to the learning rate.
Uses information from errors: The magnitude and direction of the error influence the update.
Overall optimization: The cost is calculated based on the training samples rather than simply counting classification mistakes.
Less abrupt changes: Small changes in the weights generally produce small changes in the cost.

In contrast, the Perceptron mainly records correct/incorrect classifications, which makes its mistakes history more discrete and potentially irregular.

Conclusion for the convergence plot

The Perceptron mistakes curve shows how the number of classification errors changes during training, whereas the Gradient Descent cost curve shows how the overall loss decreases. Therefore, Gradient Descent generally produces a smoother convergence curve because it optimizes a continuous cost function through gradual weight updates.

2. Accuracy Report

The second task is to calculate and compare the Test Accuracy of all three models.

Test accuracy measures how well a trained model performs on unseen test data.

The formula is:

Test Accuracy = (Number of correctly classified test samples / Total number of test samples) × 100

For example, if a model correctly predicts 95 samples out of 100 test samples:

Test Accuracy = (95 / 100) × 100 = 95%

The accuracy of all three models should be calculated using the same test dataset so that the comparison is fair.
Analysis

After calculating the three values, the model with the highest test accuracy can be considered the best-performing model in terms of classification accuracy on the given test set.

then the Margin Model would have the best test accuracy because it correctly classified the largest percentage of unseen samples.

However, accuracy should not always be the only criterion for selecting a model. In a real-world water-quality classification problem, the consequences of an incorrect prediction may be important. Therefore, factors such as the decision boundary, margin, false positives, and false negatives should also be considered.

Overall interpretation
If Perceptron has the highest accuracy, it performed best on this particular test dataset.
If Gradient Descent has the highest accuracy, its optimized weights provided better generalization.
If the Margin Model has the highest accuracy, its larger separation between classes may have helped it generalize better.

The actual values should be filled in using the accuracy obtained from the experiment rather than assuming particular results.

3. Safety Margin

If a new water sample has chemical levels that are very close to the decision boundary, I would trust the Margin-based model more than the Perceptron, especially when making a decision where uncertainty and safety are important.

Why?

The decision boundary separates the two classes. For example, the model might classify water samples into:

Safe
Unsafe

A sample that is far away from the decision boundary is relatively easy to classify because the model has greater confidence in its prediction.

However, a sample that lies very close to the boundary is more difficult to classify. A small change in its chemical measurements could potentially move it from one class to another.

Perceptron

The Perceptron primarily focuses on finding a decision boundary that correctly classifies the training examples.

Its main objective is essentially:

Correctly classify the training samples.

It does not explicitly try to maximize the distance between the decision boundary and the training samples.

Therefore, two different classes could potentially lie quite close to the Perceptron decision boundary.

Margin-based model

A margin-based model attempts to create a larger separation or safety margin between the decision boundary and the closest training samples.

This is particularly useful when a new sample lies close to the boundary.

The idea can be visualized as:

Class A → | margin | → decision boundary ← | margin | ← Class B

A larger margin means that the decision boundary has more separation from the closest examples.

Why the margin is important for water classification

Suppose we are classifying water into safe and unsafe categories based on chemical properties.

Imagine two samples:

Sample A: Very far from the decision boundary.
Sample B: Very close to the decision boundary.

Sample A is easier to classify because even if there is a small measurement error, its classification may remain unchanged.

Sample B is more uncertain because a small variation in chemical concentration or measurement error could cause the classification to change.

Therefore, when dealing with a sample close to the boundary, a model that considers margin and separation provides a more robust classification approach.
then the Margin Model would have the best test accuracy because it correctly classified the largest percentage of unseen samples.

However, accuracy should not always be the only criterion for selecting a model. In a real-world water-quality classification problem, the consequences of an incorrect prediction may be important. Therefore, factors such as the decision boundary, margin, false positives, and false negatives should also be considered.

Overall interpretation
If Perceptron has the highest accuracy, it performed best on this particular test dataset.
If Gradient Descent has the highest accuracy, its optimized weights provided better generalization.
If the Margin Model has the highest accuracy, its larger separation between classes may have helped it generalize better.

The actual values should be filled in using the accuracy obtained from the experiment rather than assuming particular results.

3. Safety Margin

If a new water sample has chemical levels that are very close to the decision boundary, I would trust the Margin-based model more than the Perceptron, especially when making a decision where uncertainty and safety are important.

Why?

The decision boundary separates the two classes. For example, the model might classify water samples into:

Safe
Unsafe

A sample that is far away from the decision boundary is relatively easy to classify because the model has greater confidence in its prediction.

However, a sample that lies very close to the boundary is more difficult to classify. A small change in its chemical measurements could potentially move it from one class to another.

Perceptron

The Perceptron primarily focuses on finding a decision boundary that correctly classifies the training examples.

Its main objective is essentially:

Correctly classify the training samples.

It does not explicitly try to maximize the distance between the decision boundary and the training samples.

Therefore, two different classes could potentially lie quite close to the Perceptron decision boundary.

Margin-based model

A margin-based model attempts to create a larger separation or safety margin between the decision boundary and the closest training samples.

This is particularly useful when a new sample lies close to the boundary.

The idea can be visualized as:

Class A → | margin | → decision boundary ← | margin | ← Class B

A larger margin means that the decision boundary has more separation from the closest examples.

Why the margin is important for water classification

Suppose we are classifying water into safe and unsafe categories based on chemical properties.

Imagine two samples:

Sample A: Very far from the decision boundary.
Sample B: Very close to the decision boundary.

Sample A is easier to classify because even if there is a small measurement error, its classification may remain unchanged.

Sample B is more uncertain because a small variation in chemical concentration or measurement error could cause the classification to change.

Therefore, when dealing with a sample close to the boundary, a model that considers margin and separation provides a more robust classification approach.
Overall Critical Analysis

The three models approach classification from somewhat different perspectives.

The Perceptron is simple and computationally efficient. It updates its weights when it makes classification errors and works well when the classes are linearly separable. However, it does not explicitly optimize a smooth loss function or maximize the margin.

Gradient Descent provides a systematic optimization approach by minimizing a cost function. Because the cost changes continuously with the model parameters, its convergence curve is generally smoother. It can also provide a more controlled optimization process.

The Margin-based model goes one step further by considering not only whether the samples are correctly classified but also how far they are from the decision boundary. This can provide better robustness, especially for samples that are close to the boundary.

Final conclusion

The convergence analysis shows that Gradient Descent generally has a smoother learning curve because it minimizes a continuous cost function through gradual parameter updates. The accuracy comparison should be based on the actual test results obtained for the three models, with the model having the highest test accuracy being the best performer for that particular dataset.

For a new water sample close to the decision boundary, the Margin-based model would generally be the preferred model because it considers the separation between the classes and aims for a more robust decision boundary. This is especially important in water-quality classification, where a small change in chemical measurements could potentially change the predicted class.

# Discussion Questions

### Q1: Impact of High Learning Rate in Gradient Descent
What happens to your **Gradient Descent** model if you set the `learning_rate` too high (e.g., `1.0`)?
*Hint: Think about convergence, overshooting, and divergence.*

---

### Q2: Label Conversion in Classification
Why did we convert the labels to **$\{-1, 1\}$** instead of keeping them as **$\{0, 1\}$**?
*Hint: Consider the mathematical formulation of the loss function (e.g., Hinge Loss) and symmetry.*

---

### Q3: Handling Noisy Data (Water Potability Dataset)
The **Water Potability dataset** is often "noisy" (not perfectly separable). Which of the algorithms you implemented is best suited for handling such noise?
*Hint: Think about robustness to outliers and margin-based classifiers.*


Q1: Impact of High Learning Rate in Gradient Descent

If the learning rate is set too high, for example learning_rate = 1.0, Gradient Descent may fail to converge properly.

The learning rate determines how large a step the model takes while updating its weights. A small learning rate makes gradual updates, while a large learning rate makes much larger updates.

With a very high learning rate:

The model may overshoot the minimum of the cost function.
Instead of gradually decreasing, the cost may oscillate between high and low values.
The algorithm may fail to converge.
In extreme cases, the cost can become extremely large, causing divergence.
The weights may become very large, resulting in unstable predictions.

For example, instead of moving toward the minimum like:

High cost → lower cost → lower cost → minimum

the model may behave like:

High cost → overshoot → higher cost → overshoot again → divergence

Therefore, a learning rate such as 1.0 may be too large depending on the dataset and feature scaling.

Conclusion

A high learning rate can cause overshooting, oscillation, and divergence, while a suitable learning rate allows Gradient Descent to converge smoothly toward the minimum.

Q2: Why Convert Labels from {0, 1} to {-1, 1}?

We convert classification labels from {0, 1} to {-1, 1} because many mathematical formulations of linear classifiers, especially Hinge Loss and margin-based classifiers, naturally use labels of -1 and +1.

For a binary classifier, the prediction can be represented using:

f(x)=w
T
x+b

The sign of this value determines the class:

Positive value → +1
Negative value → -1

The Hinge Loss is commonly written as:

L=max(0,1−yf(x))

where y must be either -1 or +1.

y=max(0,1−x)

Using -1 and +1 makes the mathematical formulation symmetric around zero.

For example:

If y = +1, the model wants f(x) to be sufficiently positive.
If y = -1, the model wants f(x) to be sufficiently negative.

This makes the concept of a margin around the decision boundary much easier to express.

Why not simply use 0 and 1?

Using 0 and 1 is perfectly valid for many classification algorithms, such as logistic regression. However, it is less convenient for the standard formulation of Hinge Loss because the negative class would not contribute symmetrically to the mathematical expression.

Conclusion

Labels are converted to {-1, +1} because:

They provide a symmetric representation.
They work naturally with Hinge Loss.
They make margin calculations easier.
They clearly represent the two sides of the decision boundary.
Q3: Handling Noisy Data in the Water Potability Dataset

The Water Potability dataset is noisy, meaning that the data is not perfectly linearly separable. There may be overlapping samples, measurement errors, outliers, or samples whose chemical properties are very similar even though their labels are different.

Among the algorithms implemented, a margin-based classifier such as SVM with a soft margin is generally better suited to handling this type of noisy data.

Why?

A simple Perceptron tries to correctly classify the training samples. If the data is not perfectly separable, the Perceptron can continue making updates because some samples may never be classified correctly.

A margin-based classifier takes a different approach. It attempts to find a decision boundary with a large margin while allowing some training samples to be misclassified when necessary.

This is particularly useful for noisy datasets.

Soft Margin

A soft-margin classifier does not require every training sample to be perfectly classified.

Instead, it allows some samples to fall inside the margin or even on the wrong side of the decision boundary.

This is useful because noisy or outlier samples should not be allowed to completely determine the decision boundary.

For example:

Without noise:

Class A |------ Margin ------| Boundary |------ Margin ------| Class B

With noisy points:

Class A |------ Margin ------| Boundary |------ Margin ------| Class B

A few unusual samples may fall inside the margin, but the model can still maintain a good overall separation between the two classes.